# Layout

> Itemized treemaps: a hierarchy of groups in, named grid cells out.

In [ ]:
#| default_exp treemap.layout


In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *
import random

An itemized treemap shows how a set splits into subsets, and subsets into sub-subsets, with
every member written out. Each item is one cell of a grid; each group, at every depth, is one
connected region of cells; and the whole map is a rectangle whose only carve-out is the end of
its last column.

## How the cells are placed

Cells are laid along a **snake**: down the first column, up the second, down the third, and so
on, through a grid `h` rows tall. Every group gets one unbroken run of that path, so every
group's region is connected, at every depth, whatever the sizes.

Each group spends one to three cells on its **header** (its name). The header goes where it
reads best, tried in a fixed order:

1. a whole row at the top or bottom edge of the region, so the header has no names beside it
   (*wide*: one line across cells; *tall*: the label broken down one column, one line per cell);
2. otherwise the region's top-left cell (or, when the region starts with a stub at the foot of a
   column, the top of the column holding most of it);
3. otherwise the first cells of the region's run.

Moving a header can leave a subgroup's run touching its neighbour only at a corner, so every
choice is checked and the search backs up when needed. The last option always works, because
the remaining runs are then still pieces of the snake.

Names inside a group are filled column by column, top to bottom, in natural order, so the
reading order does not depend on which way the snake runs.

The layout depends only on integers: item counts and each header's size in cells
(`Group.header_span`). Measuring text to choose those sizes happens in
[`treemap.render.prepare`](render.html).

In [ ]:
#| export
import math
from dataclasses import dataclass, field

from itemiset.core.grid import natural_key

## Items and groups

In [ ]:
#| export
@dataclass
class Item:
    "One member of the map: its name, and optionally a value and the numeral to print before the name."
    name: str
    value: float | None = None   # an optional quantity carried along (not drawn)
    numeral: str | None = None   # optional text drawn before the name, e.g. "163"

@dataclass
class Group:
    "A group (or the root): either subgroups (`children`) or items, plus how its header is shaped."
    name: str
    label: str | None = None             # header text; defaults to `name`
    children: list = field(default_factory=list)
    items: list = field(default_factory=list)
    header_span: int = 1                 # cells the header occupies (1-3); set by `render.prepare`
    header_wide: str | None = None       # the header's text on one line, when it spans cells side by side
    header_lines: list | None = None     # the header broken over one cell per line; None if it cannot be

    @property
    def is_leaf(self) -> bool: return not self.children

    def n_items(self) -> int:
        "Items in this group, at any depth."
        return len(self.items) if self.is_leaf else sum(c.n_items() for c in self.children)

    def n_cells(self, root: bool = False) -> int:
        "Cells this group needs: its items plus every header in it (the root has no header)."
        header = 0 if root else self.header_span
        if self.is_leaf:
            return header + len(self.items)
        return header + sum(c.n_cells() for c in self.children)

def leaf_items(g: Group) -> list:
    "Every item under `g`, in tree order."
    return list(g.items) if g.is_leaf else [it for c in g.children for it in leaf_items(c)]

`build_tree` turns `(path, item)` records into a tree, where `path` names the item's group at
each level, e.g. `("Asia", "Eastern Asia")`. Subgroups smaller than `min_group` are pooled into
one "other" subgroup, and a group left with a single subgroup absorbs it. Siblings are ordered
largest first (ties in natural order); groups named in `last` go at the end, in that order.

In [ ]:
#| export
def build_tree(records, min_group: int = 1, other_label: str = "other", order: str = "size",
               last: tuple = (), labels: dict | None = None) -> Group:
    "A `Group` tree from `(path, Item)` records. `labels` maps group names to shorter header text."
    labels = labels or {}
    root = Group("root")
    for path, item in records:
        g = root
        for part in path:
            nxt = next((c for c in g.children if c.name == part), None)
            if nxt is None:
                nxt = Group(part, labels.get(part))
                g.children.append(nxt)
            g = nxt
        g.items.append(item)

    def tidy(g: Group, depth: int):
        if g.children and g.items:
            raise ValueError(f"group {g.name!r} holds both items and subgroups; put its items in a subgroup")
        if g.is_leaf:
            g.items.sort(key=lambda it: natural_key(it.name))
            return
        for c in g.children:
            tidy(c, depth + 1)
        if depth > 0 and min_group > 1:
            small = [c for c in g.children if c.n_items() < min_group]
            if len(small) > 1 or (small and len(small) < len(g.children)):
                keep = [c for c in g.children if c.n_items() >= min_group]
                pooled = Group(f"{g.name}/{other_label}", other_label,
                               items=sorted([it for c in small for it in leaf_items(c)],
                                            key=lambda it: natural_key(it.name)))
                g.children = keep + [pooled]
        if depth > 0 and len(g.children) == 1:   # a lone subgroup merges into its parent
            only = g.children[0]
            g.children, g.items = only.children, only.items
        if order == "size":
            g.children.sort(key=lambda c: (c.name in last or c.label == other_label,
                                           last.index(c.name) if c.name in last else 0,
                                           -c.n_items(), natural_key(c.name)))
    tidy(root, 0)
    return root

In [ ]:
recs = [(("Fruit", "Citrus"), Item(n)) for n in ["lime", "lemon", "orange"]] + \
       [(("Fruit", "Berry"), Item(n)) for n in ["fig"]] + \
       [(("Vegetable", "Root"), Item(n)) for n in ["carrot", "beet"]]
t = build_tree(recs, min_group=2)
test_eq([c.name for c in t.children], ["Fruit", "Vegetable"])
test_eq([c.label or c.name for c in t.children[0].children], ["Citrus", "other"])   # Berry (1 item) pooled
test_eq([it.name for it in t.children[1].items], ["beet", "carrot"])               # lone subgroup merged
test_eq(t.n_items(), 6)
test_eq(t.n_cells(root=True), 6 + 2 + 2)   # items + top-level headers + Fruit's two subgroup headers
test_fail(lambda: build_tree([(("A",), Item("x")), (("A", "B"), Item("y"))]), contains="both items and subgroups")

## The snake

In [ ]:
#| export
Cell = tuple   # (column, row), row 0 at the top

def snake(h: int, n: int) -> list:
    "The first `n` cells of a path down column 0, up column 1, down column 2, ... in a grid `h` rows tall."
    out = []
    for k in range(n):
        col, r = divmod(k, h)
        out.append((col, r if col % 2 == 0 else h - 1 - r))
    return out

def connected(cells) -> bool:
    "Whether `cells` form one piece, counting only side-by-side and above-below neighbours."
    cells = set(cells)
    if not cells:
        return True
    stack = [next(iter(cells))]
    seen = {stack[0]}
    while stack:
        c, r = stack.pop()
        for nb in ((c + 1, r), (c - 1, r), (c, r + 1), (c, r - 1)):
            if nb in cells and nb not in seen:
                seen.add(nb)
                stack.append(nb)
    return len(seen) == len(cells)

In [ ]:
test_eq(snake(3, 7), [(0, 0), (0, 1), (0, 2), (1, 2), (1, 1), (1, 0), (2, 0)])
assert connected(snake(3, 7)[2:6])          # any run of the snake is one piece
assert not connected([(0, 0), (1, 1)])      # corners do not count

## Placing headers

In [ ]:
#| export
def _top_left(cells) -> Cell:
    return min(cells)   # leftmost column, then top row

def _header_anchor(cells, h: int) -> Cell:
    "The region's top-left cell, unless its first column is a stub at the foot of a column; then the top of its main column."
    by_col = {}
    for c, r in cells:
        by_col.setdefault(c, []).append(r)
    first = min(by_col)
    major = min(by_col, key=lambda c: (-len(by_col[c]), c))
    if first == major or max(by_col[first]) < h - 1:
        return _top_left(cells)
    return (major, min(by_col[major]))

def _clean_headers(g: Group, cells: list) -> list:
    "Header shapes filling whole rows at the region's top or bottom edge: wide top, wide bottom, tall top, tall bottom."
    k = g.header_span
    by_row = {}
    for c, r in cells:
        by_row.setdefault(r, []).append(c)
    rows = sorted(by_row)
    wide, tall = [], []
    for edge in (rows, rows[::-1]):   # top edge, then bottom edge
        cs = sorted(by_row[edge[0]])
        if len(cs) == k and cs[-1] - cs[0] == k - 1:
            wide.append(tuple((c, edge[0]) for c in cs))
        rr = sorted(edge[:k])
        if (len(rr) == k and rr[-1] - rr[0] == k - 1 and all(len(by_row[r]) == 1 for r in rr)
                and len({by_row[r][0] for r in rr}) == 1 and (g.header_lines is not None or k == 1)):
            tall.append(tuple((by_row[r][0], r) for r in rr))
    return wide + tall

def _header_candidates(g: Group, cells: list, h: int) -> list:
    "Header cell sets to try, in order; only blocks of `g.header_span` connected cells."
    k, region = g.header_span, set(cells)
    out = _clean_headers(g, cells)
    for c, r in dict.fromkeys([_header_anchor(cells, h), _top_left(cells)]):
        shapes = [tuple((c + j, r) for j in range(k))]
        if g.header_lines is not None or k == 1:
            shapes.append(tuple((c, r + j) for j in range(k)))
        out += [s for s in shapes if set(s) <= region]
    out.append(tuple(cells[:k]))   # the first cells of the run: always one piece on the snake
    return [s for s in dict.fromkeys(out) if connected(s)]

## The layout

In [ ]:
#| export
@dataclass
class Placed:
    "What sits in a cell: an item, or part of a group's header."
    kind: str              # "item" or "header"
    path: tuple            # the group's path from the root
    item: Item | None = None
    group: Group | None = None
    span: tuple = ()       # for a header: all of its cells

@dataclass
class Layout:
    "A finished treemap layout: which item or header is in each cell, and each group's cells."
    h: int                 # rows
    cols: int              # columns
    cells: dict            # (col, row) -> Placed
    regions: dict          # group path -> its cells (the root's path is ())
    root: Group
    header_fallbacks: int = 0   # headers that could not go in their first-choice place
    fig: object = field(default=None, repr=False)   # the figure, once drawn

    @property
    def n_items(self) -> int: return self.root.n_items()

    def __repr__(self):
        return (f"Layout({self.n_items} items, {len(self.root.children)} top-level groups, "
                f"{self.cols} x {self.h} grid, {len(self.cells)} cells)")

    def _repr_png_(self):
        if self.fig is None: return None
        import io
        buf = io.BytesIO()
        self.fig.savefig(buf, format="png", dpi=110)
        return buf.getvalue()

In [ ]:
#| export
def _plan(g: Group, path: tuple, cells: list, root: bool, stats: dict, h: int):
    "Placements and regions for `g` on `cells`, or None if no header choice keeps every subgroup in one piece."
    regions = {path: list(cells)}
    if g.is_leaf:
        out, body = {}, sorted(cells)   # reading order: column by column, top to bottom
        if not root:
            cands = _header_candidates(g, cells, h)
            if not cands:
                return None             # the parent tries its next header choice
            hs = cands[0]
            for c in hs:
                out[c] = Placed("header", path, group=g, span=hs)
                body.remove(c)
        for cell, it in zip(body, g.items):
            out[cell] = Placed("item", path, item=it)
        return out, regions

    candidates = [None] if root else _header_candidates(g, cells, h)
    for rank, hs in enumerate(candidates):
        rest = [c for c in cells if c not in hs] if hs is not None else list(cells)
        chunks, i = [], 0
        for ch in g.children:
            n = ch.n_cells()
            chunks.append(rest[i:i + n])
            i += n
        if not all(connected(ch) for ch in chunks):
            continue
        out = {} if hs is None else {c: Placed("header", path, group=g, span=hs) for c in hs}
        regs, sub_fallbacks = dict(regions), 0
        for ch, chunk in zip(g.children, chunks):
            sub = {"fallbacks": 0}
            res = _plan(ch, path + (ch.name,), chunk, False, sub, h)
            if res is None:
                break
            out.update(res[0])
            regs.update(res[1])
            sub_fallbacks += sub["fallbacks"]
        else:
            stats["fallbacks"] += sub_fallbacks + (1 if rank > 0 else 0)
            return out, regs
    return None

The grid's height is chosen to bring the map's shape close to `target_aspect` (width over
height), given each cell's shape (`cell_aspect`), while avoiding a nearly empty last column.
Every height from 1 up is scored, the score is rounded to an integer, and the first lowest wins.

In [ ]:
#| export
def layout_tree(root: Group, cell_aspect: float, target_aspect: float = 1.6, h: int | None = None,
                waste: float = 0.5) -> Layout:
    "Lay out `root` on a grid. `h` fixes the number of rows; otherwise it is chosen from the aspects."
    n = root.n_cells(root=True)
    if n == 0:
        raise ValueError("nothing to lay out: the tree has no items")
    if h is None:
        best = None
        for cand in range(1, n + 1):
            cols = -(-n // cand)
            aspect = cols * cell_aspect / cand
            empty = cols * cand - n
            score = round((abs(math.log(aspect / target_aspect)) + waste * empty / cand) * 1e9)
            if best is None or score < best[0]:
                best = (score, cand)
        h = best[1]
    stats = {"fallbacks": 0}
    res = _plan(root, (), snake(h, n), True, stats, h)
    if res is None:   # cannot happen: first-cells headers always work
        raise RuntimeError("no valid layout")
    return Layout(h, -(-n // h), res[0], res[1], root, stats["fallbacks"])

In [ ]:
#| export
def check_layout(lay: Layout) -> dict:
    "The layout's guarantees, each `True` or `False`."
    placed = [p.item.name for p in lay.cells.values() if p.kind == "item"]
    pieces = all(connected(cells) for cells in lay.regions.values())
    headers = True
    for path, cells in lay.regions.items():
        if not path: continue
        hdr = [c for c in cells if lay.cells[c].kind == "header" and lay.cells[c].path == path]
        if not hdr or len(hdr) != lay.cells[hdr[0]].group.header_span or not connected(hdr):
            headers = False
    return {"Every item placed once": len(placed) == lay.root.n_items(),
            "One cell per item and header cell": len(lay.cells) == lay.root.n_cells(root=True),
            "Each group is one piece": pieces,
            "Each header is one block of its size": headers,
            "All cells on the grid": all(0 <= c < lay.cols and 0 <= r < lay.h for c, r in lay.cells)}

A small map, drawn as text: `#` marks header cells, letters the items.

In [ ]:
def text_grid(lay):
    rows = [["." for _ in range(lay.cols)] for _ in range(lay.h)]
    for (c, r), p in lay.cells.items():
        rows[r][c] = "#" if p.kind == "header" else p.item.name[0]
    return "\n".join("".join(r) for r in rows)

lay = layout_tree(t, cell_aspect=3, h=4)
print(lay)
print(text_grid(lay))
assert all(check_layout(lay).values())

Layout(6 items, 2 top-level groups, 3 x 4 grid, 10 cells)
#bc
###
lf.
lo.


A header can span cells. Here Fruit's header takes two cells side by side, and the map grows by one cell:

In [ ]:
t.children[0].header_span = 2
lay2 = layout_tree(t, cell_aspect=3, h=4)
print(text_grid(lay2))
assert all(check_layout(lay2).values())
t.children[0].header_span = 1

###
##b
lfc
lo.


## Randomised check

Random trees one to three levels deep, with random header sizes, laid out at every height:
every layout must keep all its guarantees.

In [ ]:
def random_tree(rng, depth, prefix="g"):
    g = Group(prefix)
    g.header_span = rng.choice((1, 1, 2, 3))
    g.header_lines = ["x"] * g.header_span if rng.random() < 0.6 else None
    if depth == 0:
        g.items = [Item(f"{prefix}i{k}") for k in range(rng.randint(1, 12))]
    else:
        g.children = [random_tree(rng, depth - 1, f"{prefix}.{k}") for k in range(rng.randint(1, 6))]
    return g

n_layouts = 0
for seed in range(60):
    rng = random.Random(seed)
    root = Group("root", children=[random_tree(rng, rng.randint(0, 2), f"t{k}") for k in range(rng.randint(1, 8))])
    for h in range(2, min(root.n_cells(root=True), 40) + 1):
        res = check_layout(layout_tree(root, cell_aspect=3, h=h))
        assert all(res.values()), (seed, h, res)
        n_layouts += 1
print(n_layouts, "random layouts, all valid")

2160 random layouts, all valid


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()